# Cross-Dataset Validation — Models on RAF-DB

This notebook performs **cross-dataset validation** by evaluating all trained models
from `best_models_output/` against the **RAF-DB** test set.

**RAF-DB** contains 7 basic emotions with numeric labels:

| Label | Emotion   | Count (test) |
|-------|-----------|-------------:|
| 1     | Surprise  | 329          |
| 2     | Fear      | 74           |
| 3     | Disgust   | 160          |
| 4     | Happy     | 1185         |
| 5     | Sad       | 478          |
| 6     | Angry     | 162          |
| 7     | Neutral   | 680          |

**What this notebook does:**
1. Loads RAF-DB test images with ground-truth labels from CSV
2. Auto-discovers all model files from `best_models_output/`
3. Handles label mapping between RAF-DB (7-class) and model outputs (5 or 7 classes)
4. Runs inference on all test images with every model
5. Computes **accuracy, confusion matrices, classification reports**
6. Per-emotion accuracy breakdown
7. Cross-model comparison
8. Grad-CAM visualizations on sampled images
9. Exports all results to `results/cross_validation_rafdb/`

## 1. Setup & Imports

In [ ]:
# Install missing packages (uncomment on Colab)
# !pip install -q torch torchvision timm opencv-python matplotlib seaborn pandas numpy scikit-learn tqdm

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
import torchvision.models as tv_models

try:
    import timm
    HAS_TIMM = True
    print("timm available:", timm.__version__)
except ImportError:
    HAS_TIMM = False
    print("timm not available — HSEmotion models will be skipped")

try:
    from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
    HAS_SKLEARN = True
    print("scikit-learn available")
except ImportError:
    HAS_SKLEARN = False
    print("scikit-learn not available")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")

## 2. Model Architectures

All four model architectures used in this project are defined below so the
notebook is **fully self-contained**.

| Architecture | Backbone | Input | Params |
|---|---|---|---|
| **MiniXception** | Custom separable-conv residual blocks | 48x48 3ch | ~85K |
| **ResNet-18** | torchvision ResNet-18 (ImageNet) | 224x224 3ch | ~11.2M |
| **EfficientNet-B0** | torchvision EfficientNet-B0 (ImageNet) | 224x224 3ch | ~4.3M |
| **HSEmotion** | timm EfficientNet-B0 (AffectNet optional) | 224x224 3ch | ~4.0M |

In [ ]:
# ═══════════════════════════════════════════════════════════════
# MiniXception
# ═══════════════════════════════════════════════════════════════

class SeparableConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, padding=1):
        super().__init__()
        self.depthwise = nn.Conv2d(in_channels, in_channels, kernel_size=kernel_size,
                                   padding=padding, groups=in_channels, bias=False)
        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        return self.bn(x)


class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.sep_conv1 = SeparableConv2d(in_channels, out_channels)
        self.sep_conv2 = SeparableConv2d(out_channels, out_channels)
        self.pool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.skip = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=2, bias=False),
            nn.BatchNorm2d(out_channels),
        )

    def forward(self, x):
        residual = x
        x = F.relu(self.sep_conv1(x))
        x = self.sep_conv2(x)
        x = self.pool(x)
        residual = self.skip(residual)
        return F.relu(x + residual)


class MiniXception(nn.Module):
    def __init__(self, num_classes, in_channels=3, dropout=0.5):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, 8, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(8), nn.ReLU(inplace=True),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(8, 8, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(8), nn.ReLU(inplace=True),
        )
        self.block1 = ResidualBlock(8, 16)
        self.block2 = ResidualBlock(16, 32)
        self.block3 = ResidualBlock(32, 64)
        self.block4 = ResidualBlock(64, 128)
        self.conv_final = nn.Sequential(SeparableConv2d(128, 256), nn.ReLU(inplace=True))
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(p=dropout)
        self.fc = nn.Linear(256, num_classes)
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight); nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.conv1(x); x = self.conv2(x)
        x = self.block1(x); x = self.block2(x); x = self.block3(x); x = self.block4(x)
        x = self.conv_final(x)
        x = self.global_avg_pool(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        return self.fc(x)


# ═══════════════════════════════════════════════════════════════
# ResNet-18
# ═══════════════════════════════════════════════════════════════

class ResNet18(nn.Module):
    def __init__(self, num_classes=5, in_channels=3, freeze_backbone=False, unfreeze_last_n=2):
        super().__init__()
        self.in_channels = in_channels
        if in_channels == 1:
            self.channel_adapter = nn.Sequential(
                nn.Conv2d(1, 3, kernel_size=1, bias=False), nn.BatchNorm2d(3)
            )
        else:
            self.channel_adapter = None
        weights = tv_models.ResNet18_Weights.IMAGENET1K_V1
        self.backbone = tv_models.resnet18(weights=weights)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Dropout(p=0.4), nn.Linear(in_features, 256), nn.ReLU(),
            nn.Dropout(p=0.3), nn.Linear(256, num_classes)
        )
        if freeze_backbone:
            self._freeze_backbone(unfreeze_last_n)

    def _freeze_backbone(self, unfreeze_last_n=2):
        for layer in [self.backbone.conv1, self.backbone.bn1,
                      self.backbone.layer1, self.backbone.layer2,
                      self.backbone.layer3, self.backbone.layer4]:
            for p in layer.parameters(): p.requires_grad = False
        residual_layers = [self.backbone.layer1, self.backbone.layer2,
                           self.backbone.layer3, self.backbone.layer4]
        for i in range(max(0, len(residual_layers) - unfreeze_last_n), len(residual_layers)):
            for p in residual_layers[i].parameters(): p.requires_grad = True
        for p in self.backbone.fc.parameters(): p.requires_grad = True

    def forward(self, x):
        if self.channel_adapter is not None:
            x = self.channel_adapter(x)
        return self.backbone(x)


# ═══════════════════════════════════════════════════════════════
# EfficientNet-B0
# ═══════════════════════════════════════════════════════════════

class EfficientNetB0(nn.Module):
    def __init__(self, num_classes=5, in_channels=1, freeze_backbone=False, unfreeze_last_n=2):
        super().__init__()
        self.in_channels = in_channels
        if in_channels == 1:
            self.channel_adapter = nn.Sequential(
                nn.Conv2d(1, 3, kernel_size=1, bias=False), nn.BatchNorm2d(3)
            )
        else:
            self.channel_adapter = None
        weights = tv_models.EfficientNet_B0_Weights.IMAGENET1K_V1
        self.backbone = tv_models.efficientnet_b0(weights=weights)
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=0.4), nn.Linear(in_features, 256), nn.ReLU(),
            nn.Dropout(p=0.3), nn.Linear(256, num_classes)
        )
        if freeze_backbone:
            self._freeze_backbone(unfreeze_last_n)

    def _freeze_backbone(self, unfreeze_last_n=2):
        for p in self.backbone.features.parameters(): p.requires_grad = False
        total_blocks = len(self.backbone.features)
        for i in range(max(0, total_blocks - unfreeze_last_n), total_blocks):
            for p in self.backbone.features[i].parameters(): p.requires_grad = True
        for p in self.backbone.classifier.parameters(): p.requires_grad = True

    def forward(self, x):
        if self.channel_adapter is not None:
            x = self.channel_adapter(x)
        return self.backbone(x)


# ═══════════════════════════════════════════════════════════════
# HSEmotion (timm-based)
# ═══════════════════════════════════════════════════════════════

class HSEmotion(nn.Module):
    def __init__(self, num_classes=5, in_channels=3, freeze_backbone=False,
                 unfreeze_last_n=2, affectnet_pretrained=False):
        super().__init__()
        self.in_channels = in_channels
        if in_channels == 1:
            self.channel_adapter = nn.Sequential(
                nn.Conv2d(1, 3, kernel_size=1, bias=False), nn.BatchNorm2d(3)
            )
        else:
            self.channel_adapter = None

        if affectnet_pretrained:
            try:
                from hsemotion.facial_emotions import HSEmotionRecognizer
                rec = HSEmotionRecognizer(model_name='enet_b0_8_best_vgaf', device='cpu')
                self.backbone = rec.model
                print("[INFO] AffectNet-pretrained weights loaded via hsemotion.")
            except Exception:
                print("[INFO] hsemotion not available, falling back to ImageNet timm weights.")
                self.backbone = timm.create_model('efficientnet_b0', pretrained=True)
        else:
            self.backbone = timm.create_model('efficientnet_b0', pretrained=True)

        in_features = self.backbone.num_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=0.4), nn.Linear(in_features, 256), nn.ReLU(),
            nn.Dropout(p=0.3), nn.Linear(256, num_classes)
        )
        self.backbone.drop_rate = 0.0

        if freeze_backbone:
            self._freeze_backbone(unfreeze_last_n)

    def _freeze_backbone(self, unfreeze_last_n=2):
        for p in self.backbone.conv_stem.parameters(): p.requires_grad = False
        for p in self.backbone.bn1.parameters(): p.requires_grad = False
        for block in self.backbone.blocks:
            for p in block.parameters(): p.requires_grad = False
        total = len(self.backbone.blocks)
        for i in range(max(0, total - unfreeze_last_n), total):
            for p in self.backbone.blocks[i].parameters(): p.requires_grad = True
        if hasattr(self.backbone, 'conv_head'):
            for p in self.backbone.conv_head.parameters(): p.requires_grad = True
        if hasattr(self.backbone, 'bn2'):
            for p in self.backbone.bn2.parameters(): p.requires_grad = True
        for p in self.backbone.classifier.parameters(): p.requires_grad = True

    def forward(self, x):
        if self.channel_adapter is not None:
            x = self.channel_adapter(x)
        return self.backbone(x)


print("All 4 model architectures defined: MiniXception, ResNet18, EfficientNetB0, HSEmotion")

## 3. Model Loading Utilities

In [ ]:
def _create_model(model_name, num_classes, **kwargs):
    """Create a model instance by architecture name."""
    name = model_name.lower()
    if name in {'mini_xception', 'mn_xception', 'mini-xception', 'mn-xception'}:
        return MiniXception(
            num_classes=num_classes,
            in_channels=kwargs.get('in_channels', 3),
            dropout=kwargs.get('dropout', 0.5),
        )
    elif name in {'efficientnet_b0', 'efficientnet-b0', 'efficientnet'}:
        return EfficientNetB0(
            num_classes=num_classes,
            in_channels=kwargs.get('in_channels', 3),
            freeze_backbone=kwargs.get('freeze_backbone', False),
            unfreeze_last_n=kwargs.get('unfreeze_last_n', 2),
        )
    elif name in {'resnet18', 'resnet-18', 'resnet'}:
        return ResNet18(
            num_classes=num_classes,
            in_channels=kwargs.get('in_channels', 3),
            freeze_backbone=kwargs.get('freeze_backbone', False),
            unfreeze_last_n=kwargs.get('unfreeze_last_n', 2),
        )
    elif name in {'hsemotion', 'hs_emotion', 'hs-emotion'}:
        if not HAS_TIMM:
            raise ImportError("timm is required for HSEmotion. pip install timm")
        return HSEmotion(
            num_classes=num_classes,
            in_channels=kwargs.get('in_channels', 3),
            freeze_backbone=kwargs.get('freeze_backbone', False),
            unfreeze_last_n=kwargs.get('unfreeze_last_n', 2),
            affectnet_pretrained=kwargs.get('affectnet_pretrained', False),
        )
    raise ValueError(f"Unknown model name: {model_name}")


def _infer_num_classes(state_dict, model_name):
    """Infer number of output classes from a raw state dict."""
    name = model_name.lower()
    key_map = {
        'efficientnet_b0': 'backbone.classifier.4.weight',
        'resnet18':        'backbone.fc.4.weight',
        'hsemotion':       'backbone.classifier.4.weight',
        'mini_xception':   'fc.weight',
    }
    key = key_map.get(name)
    if key is None:
        raise ValueError(f"Cannot infer num_classes for: {model_name}")
    if key not in state_dict:
        raise KeyError(f"Key '{key}' not found in state dict")
    return state_dict[key].shape[0]


def load_model_from_checkpoint(model_name, checkpoint_path, device=None, class_names=None):
    """Load a trained model from a checkpoint file."""
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

    if isinstance(checkpoint, dict) and 'model_state' in checkpoint:
        state_dict = checkpoint['model_state']
        if class_names is None:
            class_names = tuple(checkpoint['classes'])
    else:
        state_dict = checkpoint
        if class_names is None:
            num_classes = _infer_num_classes(state_dict, model_name)
            class_names = tuple(f'class_{i}' for i in range(num_classes))

    kwargs = {}
    name = model_name.lower()
    if name in {'efficientnet_b0', 'resnet18', 'hsemotion'}:
        has_adapter = any(k.startswith('channel_adapter') for k in state_dict)
        kwargs['in_channels'] = 1 if has_adapter else 3
        kwargs['freeze_backbone'] = False
        if name == 'hsemotion':
            kwargs['affectnet_pretrained'] = False

    model = _create_model(model_name, num_classes=len(class_names), **kwargs)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    return model, class_names


print("Model loading utilities ready.")

## 4. Configuration & RAF-DB Data Loading

RAF-DB uses numeric labels 1-7. The standard mapping is:
- 1: surprise, 2: fear, 3: disgust, 4: happy, 5: sad, 6: angry, 7: neutral

Images are pre-aligned and stored in `DATASET/test/{label_id}/`.

In [ ]:
# ════════════════════════════════════════════════════════════
# PATHS
# ════════════════════════════════════════════════════════════
PROJECT_ROOT = Path('.').resolve()
MODELS_DIR   = PROJECT_ROOT / 'best_models_output'
RAFDB_ROOT   = PROJECT_ROOT.parent / 'data' / 'raf-db'
RAFDB_IMAGES = RAFDB_ROOT / 'DATASET' / 'test'
RAFDB_LABELS = RAFDB_ROOT / 'test_labels.csv'
OUTPUT_DIR   = PROJECT_ROOT / 'results' / 'cross_validation_rafdb'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Supported extensions
MODEL_EXTENSIONS = {'.pth', '.pt'}
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}

# ════════════════════════════════════════════════════════════
# RAF-DB label mapping (standard)
# ════════════════════════════════════════════════════════════
RAFDB_LABEL_MAP = {
    1: 'surprise',
    2: 'fear',
    3: 'disgust',
    4: 'happy',
    5: 'sad',
    6: 'angry',
    7: 'neutral',
}
RAFDB_EMOTIONS = list(RAFDB_LABEL_MAP.values())  # 7 emotions

# Model class names (from training)
EMOTION_CLASSES_5 = ('angry', 'happy', 'neutral', 'sad', 'suprise')
EMOTION_CLASSES_7 = ('angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'suprise')

# ════════════════════════════════════════════════════════════
# Discover model files
# ════════════════════════════════════════════════════════════
model_files = sorted([
    f for f in MODELS_DIR.iterdir()
    if f.is_file() and f.suffix.lower() in MODEL_EXTENSIONS
])

print(f"Found {len(model_files)} model files in {MODELS_DIR.name}/")
for mf in model_files:
    print(f"  - {mf.name}  ({mf.stat().st_size / 1024 / 1024:.1f} MB)")

# ════════════════════════════════════════════════════════════
# Load RAF-DB test labels
# ════════════════════════════════════════════════════════════
labels_df = pd.read_csv(RAFDB_LABELS)
labels_df['emotion'] = labels_df['label'].map(RAFDB_LABEL_MAP)

# Build image entries: (image_path, ground_truth_emotion)
image_entries = []
for _, row in labels_df.iterrows():
    img_name = row['image']
    label_id = row['label']
    emotion = row['emotion']
    img_path = RAFDB_IMAGES / str(label_id) / img_name
    if img_path.exists():
        image_entries.append((img_path, emotion))
    else:
        # Some datasets may not have subfolders
        alt_path = RAFDB_IMAGES / img_name
        if alt_path.exists():
            image_entries.append((alt_path, emotion))

print(f"\nLoaded {len(image_entries)} RAF-DB test images")
print(f"\nEmotion distribution:")
for emotion in RAFDB_EMOTIONS:
    count = sum(1 for _, e in image_entries if e == emotion)
    print(f"  {emotion:>10s}: {count} images")

print(f"\nOutput directory: {OUTPUT_DIR}")

## 5. Image Preprocessing

RAF-DB images are already aligned, so we load them directly without face detection.
We still provide the face-detection fallback for robustness.

In [ ]:
def _imread_unicode(path):
    """Read an image from a path that may contain non-ASCII characters."""
    path = str(path)
    buf = np.fromfile(path, dtype=np.uint8)
    img = cv2.imdecode(buf, cv2.IMREAD_COLOR)
    return img


def load_image_rgb(img_path):
    """Load image and convert to RGB. RAF-DB images are already aligned."""
    img_bgr = _imread_unicode(img_path)
    if img_bgr is None:
        return None
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)


def build_eval_transform(input_size, grayscale):
    """Build evaluation transform matching training pipeline."""
    t = [transforms.Resize((input_size, input_size))]
    if grayscale:
        t.append(transforms.Grayscale(num_output_channels=3))
    t += [transforms.ToTensor(),
          transforms.Normalize(mean=[0.485, 0.456, 0.406],
                               std=[0.229, 0.224, 0.225])]
    return transforms.Compose(t)


# ── Load all RAF-DB test images ──
face_crops = {}       # img_key -> RGB numpy array
ground_truths = {}    # img_key -> ground truth emotion label

print("Loading RAF-DB test images...")
skipped = 0
for img_path, gt_emotion in tqdm(image_entries, desc="Loading images"):
    img_key = f"{gt_emotion}/{img_path.name}"
    img_rgb = load_image_rgb(img_path)
    if img_rgb is not None:
        face_crops[img_key] = img_rgb
        ground_truths[img_key] = gt_emotion
    else:
        skipped += 1

print(f"\nLoaded {len(face_crops)}/{len(image_entries)} images successfully")
if skipped > 0:
    print(f"Skipped {skipped} images (could not read)")
print(f"Ground truth distribution: {dict(pd.Series(list(ground_truths.values())).value_counts().sort_index())}")

### Display Sample Images (5 per emotion)

In [ ]:
n_samples_per_emotion = 5
fig, axes = plt.subplots(len(RAFDB_EMOTIONS), n_samples_per_emotion,
                         figsize=(3 * n_samples_per_emotion, 3.5 * len(RAFDB_EMOTIONS)))

for row_idx, emotion in enumerate(RAFDB_EMOTIONS):
    emotion_keys = [k for k, v in ground_truths.items() if v == emotion]
    samples = emotion_keys[:n_samples_per_emotion]
    for col_idx in range(n_samples_per_emotion):
        ax = axes[row_idx, col_idx]
        if col_idx < len(samples):
            ax.imshow(face_crops[samples[col_idx]])
            ax.set_title(samples[col_idx].split('/')[-1], fontsize=7)
        ax.axis('off')
    axes[row_idx, 0].set_ylabel(emotion, fontsize=12, rotation=0, labelpad=60, va='center')

plt.suptitle("RAF-DB Test Set Samples (per emotion)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Model Loading & Evaluation

For each model:
1. Auto-detect architecture and dataset from filename
2. Load weights and run inference on all RAF-DB test images
3. Handle label mapping between model classes and RAF-DB classes
4. For 5-class models, only evaluate on the 5 overlapping emotions

In [ ]:
def detect_model_arch(filename):
    """Detect model architecture from filename."""
    name = filename.lower()
    if 'hsemotion' in name:   return 'hsemotion'
    if 'efficientnet' in name: return 'efficientnet_b0'
    if 'resnet' in name:       return 'resnet18'
    if 'xception' in name:     return 'mini_xception'
    return None


def detect_dataset(filename):
    """Detect dataset from filename."""
    return 'raf_db' if ('rafdb' in filename.lower() or 'raf_db' in filename.lower()) else 'ferplus'


def get_input_config(model_arch, dataset):
    """Return (input_size, grayscale) for a model/dataset combo."""
    input_size = 48 if model_arch == 'mini_xception' else 224
    grayscale = (dataset == 'ferplus')
    return input_size, grayscale


def get_class_names_for_count(n):
    if n == 5: return EMOTION_CLASSES_5
    if n == 7: return EMOTION_CLASSES_7
    return tuple(f'class_{i}' for i in range(n))


def normalize_emotion(emotion):
    """Normalize emotion name for comparison (handles suprise/surprise typo)."""
    e = emotion.lower().strip()
    if e == 'suprise':
        return 'surprise'
    return e


# ════════════════════════════════════════════════════════════
# Main evaluation loop
# ════════════════════════════════════════════════════════════
loaded_models = {}   # model_name -> (model, class_names, arch, input_size, grayscale)
all_results = []     # list of dicts

print("=" * 70)
print("CROSS-DATASET VALIDATION: Models on RAF-DB")
print("=" * 70)

for model_path in model_files:
    model_name = model_path.stem
    print(f"\n{'─' * 60}")
    print(f"Loading model: {model_name}")
    print(f"{'─' * 60}")

    try:
        arch = detect_model_arch(model_path.name)
        if arch is None:
            print(f"  [SKIP] Cannot detect architecture for: {model_path.name}")
            continue
        if arch == 'hsemotion' and not HAS_TIMM:
            print(f"  [SKIP] timm not installed, cannot load HSEmotion model")
            continue

        dataset = detect_dataset(model_path.name)
        input_size, grayscale = get_input_config(arch, dataset)

        print(f"  Architecture : {arch}")
        print(f"  Trained on   : {dataset}")
        print(f"  Input size   : {input_size}x{input_size}")
        print(f"  Grayscale    : {grayscale}")

        model, class_names_raw = load_model_from_checkpoint(
            model_name=arch,
            checkpoint_path=model_path,
            device=DEVICE,
            class_names=get_class_names_for_count(5),
        )

        if class_names_raw[0].startswith('class_'):
            class_names = get_class_names_for_count(len(class_names_raw))
        else:
            class_names = class_names_raw

        # Normalized class names for matching
        class_names_norm = [normalize_emotion(c) for c in class_names]
        num_model_classes = len(class_names)

        print(f"  Classes      : {class_names}")
        print(f"  Parameters   : {sum(p.numel() for p in model.parameters()):,}")

        loaded_models[model_name] = (model, class_names, arch, input_size, grayscale)

        eval_transform = build_eval_transform(input_size, grayscale)
        model.eval()

        # Determine which RAF-DB emotions this model can predict
        model_can_predict = set(class_names_norm)
        skipped_count = 0

        for img_key, face_rgb in tqdm(face_crops.items(), desc=f"  Evaluating {model_name}"):
            gt_emotion = ground_truths[img_key]
            gt_norm = normalize_emotion(gt_emotion)

            # For 5-class models, skip images whose GT is not in model classes
            if gt_norm not in model_can_predict:
                skipped_count += 1
                continue

            try:
                face_pil = Image.fromarray(face_rgb)
                inp = eval_transform(face_pil).unsqueeze(0).to(DEVICE)

                with torch.no_grad():
                    out = model(inp)
                    probs = torch.softmax(out, dim=1)[0].cpu().numpy()

                pred_idx = int(np.argmax(probs))
                pred_emotion = class_names[pred_idx]
                pred_norm = normalize_emotion(pred_emotion)

                correct = (pred_norm == gt_norm)

                result = {
                    'model': model_name,
                    'image': img_key,
                    'image_file': img_key.split('/')[-1],
                    'ground_truth': gt_emotion,
                    'ground_truth_norm': gt_norm,
                    'predicted_emotion': pred_emotion,
                    'predicted_norm': pred_norm,
                    'confidence': float(probs[pred_idx]),
                    'correct': correct,
                    'num_model_classes': num_model_classes,
                }
                for i, cls in enumerate(class_names):
                    result[f'prob_{cls}'] = float(probs[i])
                all_results.append(result)

            except Exception as e:
                pass  # Silently skip errors for large-scale evaluation

        model_results = [r for r in all_results if r['model'] == model_name]
        if model_results:
            acc = sum(r['correct'] for r in model_results) / len(model_results)
            print(f"  [OK] {model_name}: {len(model_results)} predictions, accuracy={acc:.1%}")
        if skipped_count > 0:
            print(f"  [INFO] Skipped {skipped_count} images (emotions not in model classes)")

    except Exception as e:
        print(f"  [ERROR] {model_name}: {e}")
        import traceback; traceback.print_exc()

results_df = pd.DataFrame(all_results)
print(f"\n{'=' * 70}")
print(f"Done: {len(results_df)} total predictions from {len(loaded_models)} models")
print(f"{'=' * 70}")
display(results_df.head(20))

## 7. Accuracy & Confusion Matrices

Compute per-model:
- Overall accuracy
- Confusion matrix
- Classification report (precision, recall, F1)

In [ ]:
if len(results_df) > 0:
    print("=" * 70)
    print("ACCURACY & CONFUSION MATRICES")
    print("=" * 70)

    # ── Overall accuracy per model ──
    acc_rows = []
    for mname in results_df['model'].unique():
        mdata = results_df[results_df['model'] == mname]
        acc = mdata['correct'].mean()
        n_correct = mdata['correct'].sum()
        n_total = len(mdata)
        n_classes = mdata['num_model_classes'].iloc[0]
        acc_rows.append({
            'Model': mname,
            'Model Classes': n_classes,
            'Evaluated On': n_total,
            'Accuracy': f"{acc:.1%}",
            'Correct': f"{n_correct}/{n_total}",
            'Avg Confidence': f"{mdata['confidence'].mean():.4f}",
        })

    acc_df = pd.DataFrame(acc_rows).sort_values('Accuracy', ascending=False)
    print("\nOverall Accuracy Ranking:")
    display(acc_df)

    # ── Accuracy bar chart ──
    fig, ax = plt.subplots(figsize=(10, 5))
    model_names = [r['Model'] for r in acc_rows]
    accuracies = [results_df[results_df['model'] == m]['correct'].mean() for m in model_names]
    colors = sns.color_palette('Set2', len(model_names))
    bars = ax.bar(model_names, accuracies, color=colors)
    ax.set_ylabel('Accuracy')
    ax.set_title('Cross-Dataset Validation: Model Accuracy on RAF-DB Test Set', fontsize=13)
    ax.set_ylim(0, 1.05)
    for bar, acc_val in zip(bars, accuracies):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f"{acc_val:.1%}", ha='center', fontsize=10, fontweight='bold')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    fig.savefig(str(OUTPUT_DIR / 'accuracy_comparison_rafdb.png'), dpi=150, bbox_inches='tight')
    plt.show()

    # ── Confusion matrix per model ──
    for mname in results_df['model'].unique():
        mdata = results_df[results_df['model'] == mname]
        y_true = list(mdata['ground_truth_norm'])
        y_pred = list(mdata['predicted_norm'])
        labels = sorted(set(y_true + y_pred))

        if HAS_SKLEARN:
            cm = confusion_matrix(y_true, y_pred, labels=labels)
            fig, ax = plt.subplots(figsize=(8, 7))
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels,
                        yticklabels=labels, ax=ax, linewidths=0.5)
            ax.set_xlabel('Predicted')
            ax.set_ylabel('Ground Truth')
            ax.set_title(f"Confusion Matrix (RAF-DB) — {mname}\nAccuracy: {mdata['correct'].mean():.1%}", fontsize=12)
            plt.tight_layout()
            fig.savefig(str(OUTPUT_DIR / f'confusion_matrix_{mname}.png'), dpi=150, bbox_inches='tight')
            plt.show()

            print(f"\nClassification Report — {mname}:")
            print(classification_report(y_true, y_pred, labels=labels, zero_division=0))
        else:
            print(f"\n{mname}: sklearn not available for confusion matrix")

### Per-Emotion Accuracy Breakdown

In [ ]:
if len(results_df) > 0:
    emotion_acc = []
    for mname in results_df['model'].unique():
        mdata = results_df[results_df['model'] == mname]
        for gt_em in sorted(mdata['ground_truth_norm'].unique()):
            em_data = mdata[mdata['ground_truth_norm'] == gt_em]
            if len(em_data) > 0:
                emotion_acc.append({
                    'Model': mname,
                    'Emotion': gt_em,
                    'Accuracy': em_data['correct'].mean(),
                    'N': len(em_data),
                })

    if emotion_acc:
        eacc_df = pd.DataFrame(emotion_acc)
        pivot_acc = eacc_df.pivot(index='Model', columns='Emotion', values='Accuracy')

        fig, ax = plt.subplots(figsize=(12, max(3, len(loaded_models) * 0.8)))
        sns.heatmap(pivot_acc, annot=True, fmt='.1%', cmap='RdYlGn', vmin=0, vmax=1,
                    ax=ax, linewidths=0.5, cbar_kws={'label': 'Accuracy'})
        ax.set_title("Per-Emotion Accuracy by Model (RAF-DB Cross-Validation)", fontsize=13)
        ax.set_ylabel("Model"); ax.set_xlabel("Emotion (Ground Truth)")
        plt.tight_layout()
        fig.savefig(str(OUTPUT_DIR / 'per_emotion_accuracy_rafdb.png'), dpi=150, bbox_inches='tight')
        plt.show()

        # Also show the count table
        pivot_n = eacc_df.pivot(index='Model', columns='Emotion', values='N')
        print("\nSample counts per model per emotion:")
        display(pivot_n)

## 8. Cross-Model Comparison

- Agreement analysis across models
- Per-model confidence distribution
- Comparative accuracy visualization

In [ ]:
if len(results_df) > 0 and len(loaded_models) > 1:
    # Find images evaluated by all models (intersection)
    model_names_list = list(loaded_models.keys())
    image_sets = [set(results_df[results_df['model'] == m]['image']) for m in model_names_list]
    common_images = set.intersection(*image_sets) if image_sets else set()

    print(f"Images evaluated by all {len(model_names_list)} models: {len(common_images)}")

    if common_images:
        common_df = results_df[results_df['image'].isin(common_images)]
        pivot_pred = common_df.pivot(index='image', columns='model', values='predicted_norm')
        pivot_conf = common_df.pivot(index='image', columns='model', values='confidence')
        pivot_correct = common_df.pivot(index='image', columns='model', values='correct')

        # Agreement analysis
        agree_count = 0
        disagree_count = 0
        all_correct_count = 0

        for img in common_images:
            preds = pivot_pred.loc[img].dropna().values
            corrects = pivot_correct.loc[img].dropna().values
            if len(set(preds)) == 1:
                agree_count += 1
                if all(corrects):
                    all_correct_count += 1
            else:
                disagree_count += 1

        total = agree_count + disagree_count
        print(f"\nCross-Model Agreement Analysis:")
        print(f"  Agreement rate  : {agree_count}/{total} ({agree_count/total:.1%})")
        print(f"  All correct     : {all_correct_count}/{total} ({all_correct_count/total:.1%})")
        print(f"  Disagreements   : {disagree_count}/{total} ({disagree_count/total:.1%})")

    # ── Confidence distribution per model ──
    fig, ax = plt.subplots(figsize=(10, 5))
    for mname in model_names_list:
        mdata = results_df[results_df['model'] == mname]
        ax.hist(mdata['confidence'], bins=50, alpha=0.5, label=mname)
    ax.set_xlabel('Confidence')
    ax.set_ylabel('Count')
    ax.set_title('Confidence Distribution per Model (RAF-DB)', fontsize=13)
    ax.legend()
    plt.tight_layout()
    fig.savefig(str(OUTPUT_DIR / 'confidence_distribution_rafdb.png'), dpi=150, bbox_inches='tight')
    plt.show()

    # ── Confidence for correct vs incorrect predictions ──
    fig, axes = plt.subplots(1, len(model_names_list), figsize=(5 * len(model_names_list), 4))
    if len(model_names_list) == 1:
        axes = [axes]
    for ax, mname in zip(axes, model_names_list):
        mdata = results_df[results_df['model'] == mname]
        correct_conf = mdata[mdata['correct']]['confidence']
        wrong_conf = mdata[~mdata['correct']]['confidence']
        ax.hist(correct_conf, bins=30, alpha=0.6, label='Correct', color='green')
        ax.hist(wrong_conf, bins=30, alpha=0.6, label='Wrong', color='red')
        ax.set_title(f"{mname}\nAcc={mdata['correct'].mean():.1%}", fontsize=10)
        ax.set_xlabel('Confidence')
        ax.legend(fontsize=8)
    plt.suptitle('Confidence: Correct vs Wrong Predictions (RAF-DB)', fontsize=13)
    plt.tight_layout()
    fig.savefig(str(OUTPUT_DIR / 'confidence_correct_vs_wrong_rafdb.png'), dpi=150, bbox_inches='tight')
    plt.show()

## 9. Grad-CAM Visualization (Sampled Images)

Since RAF-DB test set is large (3068 images), we sample a few images per emotion
for Grad-CAM visualization.

In [ ]:
class GradCAM:
    """Grad-CAM via forward/backward hooks on a target layer."""

    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        self._fwd = target_layer.register_forward_hook(self._save_act)
        self._bwd = target_layer.register_full_backward_hook(self._save_grad)

    def _save_act(self, m, i, o):  self.activations = o.detach()
    def _save_grad(self, m, gi, go): self.gradients = go[0].detach()

    def generate(self, input_tensor, target_class=None):
        self.model.eval()
        inp = input_tensor.clone().detach().requires_grad_(True)
        out = self.model(inp)
        probs = torch.softmax(out, dim=1)[0]
        if target_class is None:
            target_class = out.argmax(dim=1).item()

        self.model.zero_grad()
        one_hot = torch.zeros_like(out)
        one_hot[0, target_class] = 1.0
        out.backward(gradient=one_hot, retain_graph=True)

        w = self.gradients.mean(dim=[2, 3], keepdim=True)
        cam = F.relu((w * self.activations).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=inp.shape[2:], mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        mn, mx = cam.min(), cam.max()
        cam = (cam - mn) / (mx - mn + 1e-8) if mx - mn > 1e-8 else np.zeros_like(cam)
        return cam, probs.detach().cpu().numpy(), target_class

    def remove(self):
        self._fwd.remove(); self._bwd.remove()


def get_target_layer(model, arch):
    """Return the last conv-like layer for Grad-CAM."""
    if arch == 'resnet18':        return model.backbone.layer4
    if arch == 'efficientnet_b0': return model.backbone.features[-1]
    if arch == 'mini_xception':   return model.conv_final
    if arch == 'hsemotion':
        if hasattr(model.backbone, 'conv_head'): return model.backbone.conv_head
        if hasattr(model.backbone, 'blocks'):    return model.backbone.blocks[-1]
    raise ValueError(f"No target layer for: {arch}")


print("Grad-CAM utilities defined.")

In [ ]:
# Sample images for Grad-CAM visualization
N_SAMPLES_GRADCAM = 3  # per emotion

sampled_keys = []
for emotion in RAFDB_EMOTIONS:
    emotion_keys = [k for k, v in ground_truths.items() if v == emotion]
    np.random.seed(42)
    selected = list(np.random.choice(emotion_keys, size=min(N_SAMPLES_GRADCAM, len(emotion_keys)), replace=False))
    sampled_keys.extend(selected)

print(f"Sampled {len(sampled_keys)} images for Grad-CAM visualization")

gradcam_results = {}   # (model, img_key) -> cam array

for model_name, (model, class_names, arch, input_size, grayscale) in loaded_models.items():
    print(f"\nModel: {model_name} (arch={arch})")
    transform = build_eval_transform(input_size, grayscale)
    class_names_norm = [normalize_emotion(c) for c in class_names]

    try:
        target_layer = get_target_layer(model, arch)
    except Exception as e:
        print(f"  [ERROR] target layer: {e}"); continue

    for img_key in sampled_keys:
        gt_emotion = ground_truths[img_key]
        gt_norm = normalize_emotion(gt_emotion)
        if gt_norm not in class_names_norm:
            continue

        face_rgb = face_crops[img_key]
        try:
            inp = transform(Image.fromarray(face_rgb)).unsqueeze(0).to(DEVICE)
            gc = GradCAM(model, target_layer)
            cam, probs, pred_idx = gc.generate(inp)
            gc.remove()
            gradcam_results[(model_name, img_key)] = cam

            pred_emotion = class_names[pred_idx]
            pred_norm = normalize_emotion(pred_emotion)
            pred_conf = probs[pred_idx]
            correct = "CORRECT" if pred_norm == gt_norm else "WRONG"

            face_rs = cv2.resize(face_rgb, (cam.shape[1], cam.shape[0]))
            hm = cv2.applyColorMap((cam * 255).astype(np.uint8), cv2.COLORMAP_JET)
            hm_rgb = cv2.cvtColor(hm, cv2.COLOR_BGR2RGB)
            overlay = cv2.addWeighted(face_rs, 0.6, hm_rgb, 0.4, 0)

            fig, axes = plt.subplots(1, 3, figsize=(15, 4))
            axes[0].imshow(face_rs);       axes[0].set_title("Original"); axes[0].axis('off')
            axes[1].imshow(overlay);       axes[1].set_title("Grad-CAM Overlay"); axes[1].axis('off')
            bar_colors = ['#e74c3c' if i == pred_idx else '#3498db' for i in range(len(class_names))]
            axes[2].barh(list(class_names), probs, color=bar_colors)
            axes[2].set_xlim(0, 1); axes[2].set_xlabel("Probability"); axes[2].set_title("Emotion Probabilities")
            title_color = 'green' if correct == "CORRECT" else 'red'
            fig.suptitle(f"[{correct}] Model: {model_name} | GT: {gt_emotion} | Pred: {pred_emotion} ({pred_conf:.1%})",
                         fontsize=12, fontweight='bold', color=title_color)
            plt.tight_layout(); plt.show()

        except Exception as e:
            print(f"    [ERROR] {img_key}: {e}")

print(f"\nGrad-CAM computed for {len(gradcam_results)} model-image pairs.")

### Cross-Model Grad-CAM Comparison

In [ ]:
if len(loaded_models) > 1 and gradcam_results:
    mnames = list(loaded_models.keys())
    for iname in sampled_keys:
        avail = [m for m in mnames if (m, iname) in gradcam_results]
        if len(avail) < 2: continue

        fig, axes = plt.subplots(1, len(avail) + 1, figsize=(4 * (len(avail) + 1), 4))
        axes[0].imshow(face_crops[iname])
        axes[0].set_title(f"Original\nGT: {ground_truths.get(iname, '?')}", fontsize=10)
        axes[0].axis('off')

        for idx, mn in enumerate(avail):
            cam = gradcam_results[(mn, iname)]
            frs = cv2.resize(face_crops[iname], (cam.shape[1], cam.shape[0]))
            hm = cv2.applyColorMap((cam * 255).astype(np.uint8), cv2.COLORMAP_JET)
            hm_rgb = cv2.cvtColor(hm, cv2.COLOR_BGR2RGB)
            ov = cv2.addWeighted(frs, 0.6, hm_rgb, 0.4, 0)
            axes[idx + 1].imshow(ov)
            axes[idx + 1].set_title(f"{mn}", fontsize=9)
            axes[idx + 1].axis('off')

        fig.suptitle(f"Cross-Model Grad-CAM: {iname}", fontsize=11, fontweight='bold')
        plt.tight_layout(); plt.show()

## 10. Aggregate Statistics

In [ ]:
if len(results_df) > 0:
    print("=" * 70)
    print("AGGREGATE STATISTICS (RAF-DB Cross-Validation)")
    print("=" * 70)

    # ── Prediction distribution ──
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Ground truth distribution
    gt_counts = results_df.groupby('model')['ground_truth_norm'].value_counts().unstack(fill_value=0)
    gt_counts.iloc[0].plot(kind='bar', ax=axes[0], color=sns.color_palette('Set2'))
    axes[0].set_title("Ground Truth Distribution (RAF-DB Test)")
    axes[0].set_xlabel("Emotion"); axes[0].set_ylabel("Count")
    axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')

    # Predicted distribution (all models combined)
    results_df['predicted_norm'].value_counts().sort_index().plot(kind='bar', ax=axes[1],
                                                                   color=sns.color_palette('Set3'))
    axes[1].set_title("Predicted Distribution (All Models Combined)")
    axes[1].set_xlabel("Emotion"); axes[1].set_ylabel("Count")
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')

    plt.tight_layout()
    fig.savefig(str(OUTPUT_DIR / 'distribution_rafdb.png'), dpi=150, bbox_inches='tight')
    plt.show()

    # ── Per-model stats table ──
    stats = []
    for mname in results_df['model'].unique():
        mdata = results_df[results_df['model'] == mname]
        probs_entropy = []
        for _, row in mdata.iterrows():
            model_obj, class_names, *_ = loaded_models[mname]
            p = [row.get(f'prob_{c}', 0) for c in class_names]
            p = np.array(p) + 1e-10
            probs_entropy.append(-np.sum(p * np.log(p)))

        stats.append({
            'Model': mname,
            'N Predictions': len(mdata),
            'Accuracy': f"{mdata['correct'].mean():.1%}",
            'Avg Confidence': f"{mdata['confidence'].mean():.4f}",
            'Std Confidence': f"{mdata['confidence'].std():.4f}",
            'Avg Entropy': f"{np.mean(probs_entropy):.4f}",
        })

    stats_df = pd.DataFrame(stats)
    print("\nPer-Model Statistics:")
    display(stats_df)

## 11. Export Results

In [ ]:
def _imwrite_unicode(path, img_bgr):
    """Write an image to a path that may contain non-ASCII characters."""
    ext = Path(path).suffix
    ok, buf = cv2.imencode(ext, img_bgr)
    if ok:
        buf.tofile(str(path))


print(f"Exporting results to: {OUTPUT_DIR}")

# 1. Predictions CSV
if len(results_df) > 0:
    p = OUTPUT_DIR / 'cross_validation_results_rafdb.csv'
    results_df.to_csv(p, index=False)
    print(f"  [OK] {p.name}")

# 2. Grad-CAM overlays
gc_count = 0
for (mname, iname), cam in gradcam_results.items():
    try:
        frs = cv2.resize(face_crops[iname], (cam.shape[1], cam.shape[0]))
        hm = cv2.applyColorMap((cam * 255).astype(np.uint8), cv2.COLORMAP_JET)
        overlay = cv2.addWeighted(cv2.cvtColor(frs, cv2.COLOR_RGB2BGR), 0.6, hm, 0.4, 0)
        safe_iname = iname.replace('/', '_').replace('\\', '_')
        out_path = OUTPUT_DIR / f'gradcam_{mname}_{safe_iname}.png'
        _imwrite_unicode(out_path, overlay)
        gc_count += 1
    except Exception:
        pass

print(f"  [OK] {gc_count} Grad-CAM overlays saved")

# 3. Accuracy summary
if len(results_df) > 0:
    summary_rows = []
    for mname in results_df['model'].unique():
        mdata = results_df[results_df['model'] == mname]
        summary_rows.append({
            'model': mname,
            'accuracy': mdata['correct'].mean(),
            'n_predictions': len(mdata),
            'n_correct': int(mdata['correct'].sum()),
            'avg_confidence': mdata['confidence'].mean(),
            'num_model_classes': mdata['num_model_classes'].iloc[0],
            'dataset': 'RAF-DB (test)',
        })
    summary_df = pd.DataFrame(summary_rows)
    p = OUTPUT_DIR / 'accuracy_summary_rafdb.csv'
    summary_df.to_csv(p, index=False)
    print(f"  [OK] {p.name}")

print(f"\nAll results exported to: {OUTPUT_DIR}")

## 12. Summary

In [ ]:
print("=" * 70)
print("CROSS-DATASET VALIDATION SUMMARY")
print("=" * 70)
print(f"\nDataset          : RAF-DB (test split)")
print(f"Total images     : {len(face_crops)}")
print(f"Models evaluated : {len(loaded_models)}")
print(f"Total predictions: {len(results_df)}")
print(f"Grad-CAM maps    : {len(gradcam_results)}")
print(f"Output directory : {OUTPUT_DIR}")

if len(results_df) > 0:
    print("\n--- Per-Model Summary ---")
    for mname in results_df['model'].unique():
        mdata = results_df[results_df['model'] == mname]
        acc = mdata['correct'].mean()
        n_cls = mdata['num_model_classes'].iloc[0]
        n_eval = len(mdata)
        print(f"  {mname}:")
        print(f"    Classes       : {n_cls}")
        print(f"    Evaluated on  : {n_eval} images")
        print(f"    Accuracy      : {acc:.1%}")
        print(f"    Avg confidence: {mdata['confidence'].mean():.4f}")

    print("\n--- Cross-Dataset Observations ---")
    print("  - Models trained on FER+/custom data are evaluated on RAF-DB")
    print("  - 5-class models only evaluated on matching emotions (angry, happy, neutral, sad, surprise)")
    print("  - 7-class models evaluated on all 7 RAF-DB emotions")
    print("  - Performance drop from training data is expected (domain shift)")

print("\nDone.")